# 02 — Experimentos com dados reais: Occupancy Detection

Inclui preparação temporal e padronização, calibração dos hiperparâmetros, execução do TID-KMeans, DenStream e k-means estático, métricas, gráficos, análise de decaimento, análise exploratória complementar e comparação online/offline.

**Arquivos necessários:** `datatest.txt`, `datatraining.txt` e `datatest2.txt`, no mesmo diretório de execução (ou adapte as variáveis `ARQ_X0`, `ARQ_CAL`, `ARQ_TESTE` na primeira célula de configuração).

**Execução:** instale as bibliotecas de `requirements.txt` e execute as células em ordem. A comparação offline no fim usa variáveis calculadas nas etapas anteriores: não execute isoladamente. Nenhuma seleção de hiperparâmetros utiliza os rótulos de teste.

**Nota de implementação:** foram preservados os valores e a parametrização do notebook original, inclusive a função `config_den()`; consulte o README antes de citar a grade numérica do parâmetro de decaimento.

In [ ]:
# %% 0. Imports e configuração
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# True = grades pequenas, só para testar se o código roda de ponta a ponta
RAPIDO = False

# Caminhos dos arquivos (ajuste se necessário)
ARQ_X0 = "dados/datatest.txt"
ARQ_CAL = "dados/datatraining.txt"
ARQ_TESTE = "dados/datatest2.txt"

# Variáveis usadas na clusterização
COLS = [
    "Temperature",
    "Light",
    "CO2",
    "Humidity"
]

TAM_LOTE = 60          # 1 ponto = 1 minuto, então 60 = 1 hora
JANELA_ARI = 4320      # 3 dias (janela que sempre contém as duas classes)


In [ ]:
# %% 1. Leitura e junção dos arquivos
partes = []
for arq in [ARQ_X0, ARQ_CAL, ARQ_TESTE]:
    d = pd.read_csv(arq, index_col=0)   # 1ª coluna é um índice sem nome no cabeçalho
    d["origem"] = arq
    partes.append(d)

df = pd.concat(partes)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Formato:", df.shape, "(esperado: 20560 linhas)")
print("Ordenado no tempo:", df["date"].is_monotonic_increasing)
print("Nulos:", int(df.isna().sum().sum()))
print(df.groupby("origem")["date"].agg(["min", "max", "count"]))


In [ ]:
# %% 2. Preparação: X0, calibração, teste, scaler e tempo
base = df[df["origem"] == ARQ_X0]
cal = df[df["origem"] == ARQ_CAL]
teste = df[df["origem"] == ARQ_TESTE]

# Scaler ajustado SÓ em X0 (sem vazamento do fluxo)
scaler = StandardScaler().fit(base[COLS])

# Tempo 0 = último instante de X0. O tempo dos lotes é medido em DIAS a partir daí,
# o que mantém o decaimento do DynamicKMeans (ultimo_tempo = 0) consistente.
T0 = base["date"].iloc[-1]


def montar(d):
    """Padroniza as variáveis e acrescenta 'Grupo' (rótulo) e 'tempo' (dias desde T0)."""
    out = pd.DataFrame(scaler.transform(d[COLS]), columns=COLS, index=d.index)
    out["Grupo"] = d["Occupancy"].values
    out["tempo"] = (d["date"] - T0).dt.total_seconds().values / 86400.0
    return out


df_base = montar(base).reset_index(drop=True)
fluxo_cal = montar(cal).reset_index(drop=True)
fluxo = montar(pd.concat([cal, teste])).reset_index(drop=True)   # calibração + teste, contínuo
N_CAL = len(cal)                                                 # primeiros N_CAL pontos do fluxo

print(f"X0: {len(df_base)} | calibração: {N_CAL} | teste: {len(fluxo) - N_CAL}")
print("Primeiro tempo do fluxo (dias):", round(float(fluxo['tempo'].iloc[0]), 3))


def fatiar(f, n):
    """Divide o fluxo em lotes consecutivos de n pontos."""
    return [f.iloc[i:i + n] for i in range(0, len(f), n)]


In [ ]:
# %% 3. DynamicKMeans usado na aplicação real (implementação e avaliadores independentes)
class DynamicKMeans:

    def __init__(
        self,
        n_clusters,
        lambda_decay=0.1,
        fator_limiar=1.5
    ):

        self.n_clusters = n_clusters
        self.lambda_decay = lambda_decay
        self.fator_limiar = fator_limiar

        # Tempo do último lote processado
        self.ultimo_tempo = 0

    # ==================================================
    # FASE INICIAL: K-MEANS
    # ==================================================

    def fit_initial(self, X0):

        kmeans = KMeans(
            n_clusters=self.n_clusters,
            random_state=123,
            n_init=50
        )

        labels = kmeans.fit_predict(X0)

        self.labels_initial = labels
        self.clusters = []

        # Informações iniciais de cada cluster
        for j in range(self.n_clusters):

            pontos_cluster = X0[labels == j]

            N_j = len(pontos_cluster)

            LS_j = pontos_cluster.sum(axis=0)

            c_j = LS_j / N_j

            # Distâncias ao centróide
            distancias = np.linalg.norm(
                pontos_cluster - c_j,
                axis=1
            )

            # Limiar específico do cluster
            # Utiliza o fator definido na inicialização
            D_j = self.fator_limiar * distancias.max()

            self.clusters.append({
                "LS": LS_j,
                "N": N_j,
                "centroide": c_j,
                "limiar": D_j
            })

        # Média dos limiares iniciais
        self.limiar_medio = np.mean([
            c["limiar"] for c in self.clusters
        ])

        # A base inicial corresponde ao tempo 0
        self.ultimo_tempo = 0

    # ==================================================
    # DECAIMENTO TEMPORAL DE TODOS OS CLUSTERS
    # ==================================================

    def aplicar_decaimento(self, tempo_atual):

        # Diferença em dias desde o último lote
        delta_t = tempo_atual - self.ultimo_tempo

        if delta_t < 0:
            raise ValueError(
                "O tempo atual não pode ser menor "
                "que o tempo do último lote processado."
            )

        # Fator de decaimento
        peso = 2.0 ** (
            -self.lambda_decay * delta_t
        )

        # Aplicar decaimento em TODOS os clusters
        for cluster in self.clusters:

            cluster["LS"] *= peso
            cluster["N"] *= peso

            # Atualizar centróide
            cluster["centroide"] = (
                cluster["LS"] / cluster["N"]
            )

        # Atualiza o tempo global do fluxo
        self.ultimo_tempo = tempo_atual

    # ==================================================
    # PROCESSAR UM LOTE
    # ==================================================

    def process_batch(self, X_batch, tempo):

        # 1. Aplicar decaimento em todos os clusters
        self.aplicar_decaimento(tempo)

        labels_pred = []

        # 2. Inserir as observações do lote
        for x_t in X_batch:

            label = self.process_one(x_t)

            labels_pred.append(label)

        return np.array(labels_pred)

    # ==================================================
    # PROCESSAR UMA OBSERVAÇÃO
    # ==================================================

    def process_one(self, x_t):

        # Calcular distância aos centróides atuais
        distancias = []

        for idx, cluster in enumerate(self.clusters):

            d = np.linalg.norm(
                x_t - cluster["centroide"]
            )

            distancias.append((idx, d))

        # Ordenar pela menor distância
        distancias.sort(key=lambda x: x[1])

        # Tentar inserir no cluster mais próximo
        for idx, d in distancias:

            cluster = self.clusters[idx]

            if d <= cluster["limiar"]:

                # Acrescentar a observação
                cluster["LS"] += x_t
                cluster["N"] += 1

                # Atualizar centróide
                cluster["centroide"] = (
                    cluster["LS"] / cluster["N"]
                )

                return idx

        # Nenhum cluster aceitou a observação:
        # criar um novo cluster
        novo_cluster = {
            "LS": x_t.copy(),
            "N": 1.0,
            "centroide": x_t.copy(),
            "limiar": self.limiar_medio
        }

        self.clusters.append(novo_cluster)

        return len(self.clusters) - 1

    # ==================================================
    # FUNÇÕES AUXILIARES
    # ==================================================

    def get_centroids(self):

        return np.array([
            c["centroide"] for c in self.clusters
        ])

    def get_number_clusters(self):

        return len(self.clusters)

    def get_cluster_info(self):

        return self.clusters

# %% 3b. Funções auxiliares e de avaliação (código original + retorno das predições)
def obter_variaveis(df_):
    """Retorna somente as variáveis usadas na clusterização."""
    colunas_excluir = ["Grupo", "tempo"]
    return df_.drop(columns=colunas_excluir, errors="ignore")


def obter_centros_denstream(modelo, colunas):
    """
    Obtém os centros dos p-micro-clusters do DenStream,
    na mesma ordem das colunas usadas como atributos.
    """
    centros = []

    for mc in modelo.p_micro_clusters.values():
        centro = mc.calc_center(modelo.timestamp)

        vetor = np.array(
            [centro.get(coluna, 0.0) for coluna in colunas],
            dtype=float
        )

        centros.append(vetor)

    if len(centros) == 0:
        return np.empty((0, len(colunas)))

    return np.vstack(centros)


def alinhar_rotulos_denstream(
    centros_atuais,
    centros_anteriores,
    ids_anteriores,
    proximo_id,
    distancia_maxima
):
    """
    Tenta manter IDs de clusters consistentes entre lotes,
    associando centros atuais aos centros anteriores por
    distância mínima.

    Centros sem correspondência recebem novos IDs.
    """
    n_atual = len(centros_atuais)
    ids_atuais = np.full(n_atual, -1, dtype=int)

    if n_atual == 0:
        return ids_atuais, proximo_id

    if len(centros_anteriores) > 0:
        distancias = cdist(centros_atuais, centros_anteriores)

        linhas, colunas = linear_sum_assignment(distancias)

        for i_atual, i_anterior in zip(linhas, colunas):
            if distancias[i_atual, i_anterior] <= distancia_maxima:
                ids_atuais[i_atual] = ids_anteriores[i_anterior]

    for i in range(n_atual):
        if ids_atuais[i] == -1:
            ids_atuais[i] = proximo_id
            proximo_id += 1

    return ids_atuais, proximo_id


def predizer_por_centros(X, centros):
    """Associa cada observação ao centro mais próximo."""
    if len(centros) == 0:
        raise ValueError(
            "O DenStream não possui p-micro-clusters "
            "disponíveis para realizar a predição."
        )

    distancias = cdist(X, centros)
    return distancias.argmin(axis=1)


def avaliar_dynamic_kmeans(
    df_base, lotes, lambda_decay, fator_limiar, n_clusters=2
):
    X0 = obter_variaveis(df_base).to_numpy(dtype=float)
    y0 = df_base["Grupo"].to_numpy()

    modelo = DynamicKMeans(
        n_clusters,
        lambda_decay=lambda_decay,
        fator_limiar=fator_limiar
    )

    modelo.fit_initial(X0)

    # Avaliação da base inicial
    pred_inicial = modelo.labels_initial
    ari_inicial = adjusted_rand_score(y0, pred_inicial)

    # Armazenamento dos resultados
    resultados_lotes = []
    y_true_stream = []
    y_pred_stream = []

    y_true_global = list(y0)
    y_pred_global = list(pred_inicial)

    # Processamento dos lotes
    for i, lote in enumerate(lotes, start=1):
        X_lote = obter_variaveis(lote).to_numpy(dtype=float)
        y_lote = lote["Grupo"].to_numpy()

        tempo_lote = (
            lote["tempo"].iloc[0]
            if "tempo" in lote.columns
            else i
        )

        pred = modelo.process_batch(X_lote, tempo_lote)

        ari_lote = adjusted_rand_score(y_lote, pred)

        resultados_lotes.append({
            "Lote": i,
            "Tempo": tempo_lote,
            "ARI": ari_lote,
            "Numero_clusters": modelo.get_number_clusters()
        })

        y_true_stream.extend(y_lote)
        y_pred_stream.extend(pred)

        y_true_global.extend(y_lote)
        y_pred_global.extend(pred)

    # ARI considerando somente as observações do fluxo
    ari_stream = adjusted_rand_score(
        y_true_stream, y_pred_stream
    )

    # ARI incluindo a base inicial e todo o fluxo
    ari_global = adjusted_rand_score(
        y_true_global, y_pred_global
    )

    return {
        "ARI_inicial": ari_inicial,
        "ARI_stream": ari_stream,
        "ARI_global_com_base": ari_global,
        "ARI_lotes": resultados_lotes,
        "Numero_clusters_final": modelo.get_number_clusters(),
        # --- acrescentado: predições para avaliar por trecho ---
        "y_true_stream": np.array(y_true_stream),
        "y_pred_stream": np.array(y_pred_stream),
    }


def avaliar_denstream(df_base, lotes, config):
    from river import cluster as river_cluster   # import local: só exige river aqui

    # ------------------------------------------------------
    # 1. Inicializar e treinar com a base inicial
    # ------------------------------------------------------

    X0_df = obter_variaveis(df_base)
    colunas = list(X0_df.columns)

    X0 = X0_df.to_numpy(dtype=float)
    y0 = df_base["Grupo"].to_numpy()

    modelo = river_cluster.DenStream(
        epsilon=config["epsilon"],
        beta=config["beta"],
        mu=config["mu"],
        decaying_factor=config["decaying_factor"],
        n_samples_init=config["n_samples_init"],
        stream_speed=config["stream_speed"]
    )

    # O River recebe cada observação como um dicionário
    for registro in X0_df.to_dict(orient="records"):
        modelo.learn_one(registro)

    # Centros disponíveis após o treinamento inicial
    centros = obter_centros_denstream(modelo, colunas)

    if len(centros) == 0:
        raise ValueError(
            "O DenStream não formou p-micro-clusters "
            "após o treinamento inicial."
        )

    # IDs iniciais dos clusters
    ids_centros = np.arange(len(centros), dtype=int)
    proximo_id = len(centros)

    # Avaliação retrospectiva da base inicial
    pred_inicial = predizer_por_centros(X0, centros)
    ari_inicial = adjusted_rand_score(y0, pred_inicial)

    y_true_global = list(y0)
    y_pred_global = list(ids_centros[pred_inicial])

    y_true_stream = []
    y_pred_stream = []
    resultados_lotes = []

    # ------------------------------------------------------
    # 2. Processar o fluxo lote a lote
    # ------------------------------------------------------

    for i, lote in enumerate(lotes, start=1):

        X_lote_df = obter_variaveis(lote)
        X_lote = X_lote_df.to_numpy(dtype=float)
        y_lote = lote["Grupo"].to_numpy()

        tempo_lote = (
            lote["tempo"].iloc[0]
            if "tempo" in lote.columns
            else i
        )

        # Predizer usando os centros disponíveis ANTES
        # de aprender as observações deste lote
        pred_indices = predizer_por_centros(X_lote, centros)
        pred = ids_centros[pred_indices]

        ari_lote = adjusted_rand_score(y_lote, pred)

        y_true_stream.extend(y_lote)
        y_pred_stream.extend(pred)

        y_true_global.extend(y_lote)
        y_pred_global.extend(pred)

        # Aprender as observações do lote, uma a uma
        for registro in X_lote_df.to_dict(orient="records"):
            modelo.learn_one(registro)

        # Obter os centros atualizados para o próximo lote
        centros_novos = obter_centros_denstream(modelo, colunas)

        if len(centros_novos) > 0:

            ids_novos, proximo_id = alinhar_rotulos_denstream(
                centros_atuais=centros_novos,
                centros_anteriores=centros,
                ids_anteriores=ids_centros,
                proximo_id=proximo_id,
                distancia_maxima=config["epsilon"]
            )

            centros = centros_novos
            ids_centros = ids_novos

        # Quantidade de p-micro-clusters após aprender o lote
        resultados_lotes.append({
            "Lote": i,
            "Tempo": tempo_lote,
            "ARI": ari_lote,
            "Numero_clusters": len(modelo.p_micro_clusters)
        })

    # ------------------------------------------------------
    # 3. Calcular os resultados gerais
    # ------------------------------------------------------

    ari_stream = adjusted_rand_score(
        y_true_stream,
        y_pred_stream
    )

    ari_global = adjusted_rand_score(
        y_true_global,
        y_pred_global
    )

    return {
        "ARI_inicial": ari_inicial,
        "ARI_stream": ari_stream,
        "ARI_global_com_base": ari_global,
        "ARI_lotes": resultados_lotes,
        "Numero_clusters_final": len(modelo.p_micro_clusters),
        # --- acrescentado: predições para avaliar por trecho ---
        "y_true_stream": np.array(y_true_stream),
        "y_pred_stream": np.array(y_pred_stream),
    }


## 7. Seleção e avaliação temporal — Occupancy

Seleciona na calibração e avalia no teste.

In [ ]:
# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

SEMENTE = 123
RAPIDO = False

TAM_LOTE = 60
JANELA = 360
PASSO = 360

# Mínimo de janelas com silhueta definida.
MIN_FRAC_VALIDAS = 0.40

# Máximo de estruturas ao final da calibração.
MAX_CLUSTERS = 15

# Para não escolher valores usando o teste.
GRADE_K = [2, 3, 4]

GRADE_ALPHA = [
    0.50, 0.75, 1.00, 1.25,
    1.50, 1.70, 2.00
]

# Tempo medido em dias.
# lambda=0 significa ausência de decaimento.
GRADE_LAMBDA = [0, 0.5, 1, 2, 5]

GRADE_EPS = [0.3, 0.5, 0.75, 1.0]
GRADE_MU = [2, 5, 8]
GRADE_MEIA = [0.5, 1, 3]

if RAPIDO:
    GRADE_K = [2, 3]
    GRADE_ALPHA = [0.75, 1.5]
    GRADE_LAMBDA = [0, 1, 5]
    GRADE_EPS = [0.5, 1.0]
    GRADE_MU = [2, 5]
    GRADE_MEIA = [1]


In [ ]:
# ============================================================
# 2. PREPARAÇÃO E VERIFICAÇÕES
# ============================================================

X0 = obter_variaveis(df_base).to_numpy(float)
Xcal = obter_variaveis(fluxo_cal).to_numpy(float)
Xflux = obter_variaveis(fluxo).to_numpy(float)

ycal = fluxo_cal["Grupo"].to_numpy()
yflux = fluxo["Grupo"].to_numpy()

N_CAL = len(fluxo_cal)

Xteste = Xflux[N_CAL:]
yteste = yflux[N_CAL:]

lotes_cal = fatiar(fluxo_cal, TAM_LOTE)
lotes_flux = fatiar(fluxo, TAM_LOTE)

COLUNAS = list(obter_variaveis(df_base).columns)

assert np.isfinite(X0).all()
assert np.isfinite(Xcal).all()
assert np.isfinite(Xflux).all()
assert len(Xflux) == len(yflux)

assert np.all(
    np.diff(fluxo["tempo"].to_numpy()) >= 0
)

print("Inicialização:", len(X0))
print("Calibração:", len(Xcal))
print("Teste:", len(Xteste))
print("Variáveis:", COLUNAS)


In [ ]:
# ============================================================
# 3. FUNÇÕES DE MÉTRICAS TEMPORAIS
# ============================================================

def janelas_indices(n, janela=JANELA, passo=PASSO):
    return [
        (i, i + janela)
        for i in range(0, n - janela + 1, passo)
    ]


def sil_seguro(X, pred):
    pred = np.asarray(pred)
    n_grupos = len(np.unique(pred))

    if not 2 <= n_grupos < len(pred):
        return np.nan

    return float(silhouette_score(X, pred))


def metricas_janelas(X, y, pred):
    """
    Uma linha por janela.

    A silhueta é indefinida quando existe um
    único cluster.

    ARI informativo: somente com pelo menos
    duas classes reais e dois clusters estimados.
    """
    linhas = []

    for i, j in janelas_indices(len(y)):
        Xw = X[i:j]
        yw = y[i:j]
        pw = pred[i:j]

        n_classes = len(np.unique(yw))
        n_clusters = len(np.unique(pw))

        sil = sil_seguro(Xw, pw)

        ari = np.nan
        if n_classes >= 2 and n_clusters >= 2:
            ari = adjusted_rand_score(yw, pw)

        linhas.append({
            "inicio": i,
            "fim": j,
            "silhueta": sil,
            "ARI_informativo": ari,
            "n_classes": n_classes,
            "n_clusters": n_clusters,
            "frac_ocupada": np.mean(yw == 1),
            "sil_valida": np.isfinite(sil),
            "ari_valido": np.isfinite(ari)
        })

    return pd.DataFrame(linhas)


def resumir_janelas(X, y, pred):
    j = metricas_janelas(X, y, pred)

    return {
        "silhueta": j["silhueta"].mean(),
        "frac_sil_valida": j["sil_valida"].mean(),
        "frac_ari_valido": j["ari_valido"].mean(),
        "ARI_janelas": j["ARI_informativo"].mean(),
        "n_janelas": len(j)
    }


def selecionar_grade(tabela):
    """
    Seleção exclusivamente por critérios internos:
    silhueta, proporção de janelas válidas e
    número de estruturas.

    Não utiliza ARI nem rótulos reais.
    """
    elegiveis = tabela.loc[
        tabela["silhueta_cal"].notna()
        &
        (tabela["frac_validas"] >= MIN_FRAC_VALIDAS)
        &
        (tabela["clusters_cal"] <= MAX_CLUSTERS)
    ].copy()

    if elegiveis.empty:
        raise ValueError(
            "Nenhuma configuração satisfez os limites. "
            "Inspecione a grade antes de alterá-los."
        )

    return elegiveis.sort_values(
        [
            "silhueta_cal",
            "frac_validas",
            "clusters_cal"
        ],
        ascending=[False, False, True]
    ).iloc[0]


In [ ]:
# ============================================================
# 4. DIAGNÓSTICO DAS JANELAS REAIS
# ============================================================

print("\n=== Composição temporal da ocupação ===")

linhas = []

for trecho, y in [
    ("Calibração", ycal),
    ("Teste", yteste)
]:
    for i, j in janelas_indices(len(y)):
        valores = y[i:j]

        linhas.append({
            "trecho": trecho,
            "inicio": i,
            "fim": j,
            "frac_ocupada": np.mean(valores == 1),
            "duas_classes": len(np.unique(valores)) == 2
        })

diag_ocupacao = pd.DataFrame(linhas)

print(
    diag_ocupacao.groupby("trecho").agg(
        janelas=("duas_classes", "size"),
        frac_duas_classes=("duas_classes", "mean"),
        ocupacao_media=("frac_ocupada", "mean")
    ).round(4)
)


In [ ]:
# ============================================================
# 5. ESCOLHA DE K EM X0
# ============================================================

print("\n=== Seleção do K inicial ===")

reg_k = []

for k in GRADE_K:
    km = KMeans(
        n_clusters=k,
        n_init=50,
        random_state=SEMENTE
    )

    labels = km.fit_predict(X0)

    reg_k.append({
        "K": k,
        "silhueta_X0": silhouette_score(X0, labels),
        "inercia": km.inertia_
    })

tabela_k = pd.DataFrame(reg_k)

print(tabela_k.round(4).to_string(index=False))

K_ESC = int(
    tabela_k.sort_values(
        "silhueta_X0",
        ascending=False
    ).iloc[0]["K"]
)

print("K selecionado:", K_ESC)


In [ ]:
import itertools

# ============================================================
# 6. CALIBRAÇÃO DO DYNAMIC K-MEANS
# ============================================================

print("\n=== Calibração Dynamic K-Means ===")

reg_dyn = []

for lam, alpha in itertools.product(
    GRADE_LAMBDA,
    GRADE_ALPHA
):
    r = avaliar_dynamic_kmeans(
        df_base,
        lotes_cal,
        lambda_decay=lam,
        fator_limiar=alpha,
        n_clusters=K_ESC
    )

    pred = np.asarray(r["y_pred_stream"])

    m = resumir_janelas(Xcal, ycal, pred)

    reg_dyn.append({
        "lambda": lam,
        "alpha": alpha,
        "silhueta_cal": m["silhueta"],
        "frac_validas": m["frac_sil_valida"],
        "clusters_cal": r["Numero_clusters_final"],
        "ARI_cal_aux": r["ARI_stream"]
    })

grade_dyn = pd.DataFrame(reg_dyn)

print(
    grade_dyn.sort_values(
        "silhueta_cal",
        ascending=False
    ).round(4).to_string(index=False)
)

esc_dyn = selecionar_grade(grade_dyn)

print("\nDynamic selecionado:")
print(esc_dyn.to_string())


In [ ]:
# ============================================================
# 7. CALIBRAÇÃO DO DENSTREAM
# ============================================================

print("\n=== Calibração DenStream ===")

# Mantém a parametrização utilizada anteriormente.
# A meia-vida é aproximada em número de observações,
# pois o River usa relógio interno, não datas reais.

STREAM_SPEED = 60


def config_den(eps, mu, meia_vida):
    return {
        "epsilon": float(eps),
        "beta": 0.75,
        "mu": int(mu),
        "decaying_factor": (
            STREAM_SPEED / (1440.0 * float(meia_vida))
        ),
        "n_samples_init": min(1000, len(df_base)),
        "stream_speed": STREAM_SPEED
    }

# A calibração completa, incluindo beta, é realizada na próxima célula.


# Função disponível antes da seleção do DenStream
def selecionar_configuracao(
    tabela,
    minimo_validas=0.40,
    max_clusters=15
):

    candidatos = tabela.loc[
        tabela["silhueta_cal"].notna()
        &
        (tabela["frac_validas"] >= minimo_validas)
        &
        (tabela["clusters_cal"] <= max_clusters)
    ].copy()

    if candidatos.empty:
        raise ValueError(
            "Nenhuma configuração atende aos critérios."
        )

    candidatos = candidatos.sort_values(
        [
            "silhueta_cal",
            "frac_validas",
            "clusters_cal"
        ],
        ascending=[False, False, True]
    )

    return candidatos.iloc[0]



In [ ]:
import itertools

# ============================================================
# CALIBRAÇÃO DENSTREAM — INCLUINDO BETA
# ============================================================

import itertools
import numpy as np
import pandas as pd

GRADE_BETA = [0.25, 0.50, 0.75]

reg_den_beta = []

for eps, mu, meia, beta in itertools.product(
    GRADE_EPS, GRADE_MU, GRADE_MEIA, GRADE_BETA
):

    # Restrição usual do DenStream
    if beta * mu <= 1:
        continue

    cfg = config_den(eps, mu, meia)
    cfg["beta"] = float(beta)

    try:
        r = avaliar_denstream(
            df_base,
            lotes_cal,
            cfg
        )

        pred = np.asarray(r["y_pred_stream"])
        m = resumir_janelas(Xcal, ycal, pred)

        reg_den_beta.append({
            "epsilon": eps,
            "mu": mu,
            "beta": beta,
            "meia_vida": meia,
            "silhueta_cal": m["silhueta"],
            "frac_validas": m["frac_sil_valida"],
            "clusters_cal": r["Numero_clusters_final"],
            "ARI_cal_aux": r["ARI_stream"]
        })

    except ValueError as erro:
        print("Ignorada:", cfg, erro)

grade_den_beta = pd.DataFrame(reg_den_beta)

print("\n=== MELHORES CONFIGURAÇÕES ===")
print(
    grade_den_beta.sort_values(
        "silhueta_cal",
        ascending=False
    ).head(15).round(4).to_string(index=False)
)

# Seleção com o mesmo critério anterior
den_beta_40 = selecionar_configuracao(
    grade_den_beta,
    minimo_validas=0.40
)

print("\n=== DENSTREAM SELECIONADO ===")
print(den_beta_40.to_string())

# Atualizar as variáveis para as células seguintes
grade_den = grade_den_beta.copy()
esc_den = den_beta_40.copy()
den_40 = den_beta_40.copy()

# Comparação com a escolha anterior
print("\n=== COMPARAÇÃO ===")
print(
    "Beta escolhido:",
    float(esc_den["beta"])
)


In [ ]:
# ============================================================
# SELEÇÃO COM E SEM RESTRIÇÃO DE 40%
# ============================================================

import numpy as np
import pandas as pd



# Dynamic K-Means
dyn_40 = selecionar_configuracao(
    grade_dyn,
    minimo_validas=0.40
)

dyn_livre = selecionar_configuracao(
    grade_dyn,
    minimo_validas=0.0
)

# DenStream
den_40 = selecionar_configuracao(
    grade_den,
    minimo_validas=0.40
)

den_livre = selecionar_configuracao(
    grade_den,
    minimo_validas=0.0
)


print("\n=== Dynamic K-Means ===")

print("\nCom 40%:")
print(dyn_40.to_string())

print("\nSem restrição:")
print(dyn_livre.to_string())


print("\n=== DenStream ===")

print("\nCom 40%:")
print(den_40.to_string())

print("\nSem restrição:")
print(den_livre.to_string())


In [ ]:
# ============================================================
# 8. EXECUÇÃO FINAL — SEM USAR O TESTE NA SELEÇÃO
# ============================================================

print("\n=== Execução final ===")

km_estatico = KMeans(
    n_clusters=K_ESC,
    random_state=SEMENTE,
    n_init=50
).fit(X0)

pred_km = km_estatico.predict(Xflux)

res_dyn = avaliar_dynamic_kmeans(
    df_base,
    lotes_flux,
    lambda_decay=float(esc_dyn["lambda"]),
    fator_limiar=float(esc_dyn["alpha"]),
    n_clusters=K_ESC
)

cfg_den = config_den(
    esc_den["epsilon"],
    esc_den["mu"],
    esc_den["meia_vida"]
)
# Usar efetivamente o beta selecionado no grid de calibração.
cfg_den["beta"] = float(esc_den["beta"])

res_den = avaliar_denstream(
    df_base,
    lotes_flux,
    cfg_den
)

pred_dyn = np.asarray(res_dyn["y_pred_stream"])
pred_den = np.asarray(res_den["y_pred_stream"])

predicoes = {
    "K-Means estático": pred_km,
    "Dynamic K-Means": pred_dyn,
    "DenStream (micro)": pred_den
}


In [ ]:
# ============================================================
# 9. TABELA FINAL DE MÉTRICAS
# ============================================================

resultados = []

for nome, pred in predicoes.items():
    cal_pred = pred[:N_CAL]
    teste_pred = pred[N_CAL:]

    mc = resumir_janelas(Xcal, ycal, cal_pred)
    mt = resumir_janelas(Xteste, yteste, teste_pred)

    if nome == "K-Means estático":
        n_final = K_ESC
    elif nome == "Dynamic K-Means":
        n_final = res_dyn["Numero_clusters_final"]
    else:
        n_final = res_den["Numero_clusters_final"]

    resultados.append({
        "Método": nome,
        "Sil_cal": mc["silhueta"],
        "Sil_teste": mt["silhueta"],
        "Validas_cal": mc["frac_sil_valida"],
        "Validas_teste": mt["frac_sil_valida"],
        "ARI_cal": adjusted_rand_score(ycal, cal_pred),
        "ARI_teste": adjusted_rand_score(yteste, teste_pred),
        "ARI_janelas_teste": mt["ARI_janelas"],
        "Frac_ARI_validos": mt["frac_ari_valido"],
        "Clusters_finais": n_final
    })

tabela_final = pd.DataFrame(resultados)

print("\n=== Comparação final ===")
print(tabela_final.round(4).to_string(index=False))


In [ ]:
# ============================================================
# 10. SILHUETA NAS MESMAS JANELAS
# ============================================================

print("\n=== Silhueta em janelas comuns ===")

# Primeiro, construímos as curvas separadamente,
# pois nenhuma janela deve atravessar a divisão
# entre calibração e teste.

curvas = {}

for nome, pred in predicoes.items():

    c = metricas_janelas(
        Xcal,
        ycal,
        pred[:N_CAL]
    )

    t = metricas_janelas(
        Xteste,
        yteste,
        pred[N_CAL:]
    )

    t["inicio"] += N_CAL
    t["fim"] += N_CAL

    curvas[nome] = {
        "cal": c,
        "teste": t
    }


def comparar_janelas_comuns(trecho):
    tabelas = []

    for nome, d in curvas.items():
        z = d[trecho][
            ["inicio", "fim", "silhueta"]
        ].copy()

        z = z.rename(
            columns={"silhueta": nome}
        )

        z = z.set_index(["inicio", "fim"])
        tabelas.append(z)

    conjunto = pd.concat(tabelas, axis=1)

    comuns = conjunto.dropna()

    print(f"\n{trecho.upper()}")
    print("Janelas totais:", len(conjunto))
    print("Janelas comuns válidas:", len(comuns))

    if comuns.empty:
        print("Nenhuma janela comum válida.")
        return conjunto, comuns

    resumo = pd.DataFrame({
        "Silhueta média comum": comuns.mean(),
        "DP entre janelas": comuns.std(),
        "Mediana": comuns.median()
    })

    print(resumo.round(4).to_string())

    return conjunto, comuns


sil_cal_todas, sil_cal_comuns = (
    comparar_janelas_comuns("cal")
)

sil_teste_todas, sil_teste_comuns = (
    comparar_janelas_comuns("teste")
)

# Comparação pareada: cada diferença corresponde
# à mesma janela temporal.
if not sil_teste_comuns.empty:

    referencia = sil_teste_comuns[
        "K-Means estático"
    ]

    for nome in [
        "Dynamic K-Means",
        "DenStream (micro)"
    ]:
        diferenca = sil_teste_comuns[nome] - referencia

        print(f"\n{nome} - K-Means estático")
        print("Diferença média:", round(diferenca.mean(), 4))
        print(
            "Fração das janelas em que supera:",
            round((diferenca > 0).mean(), 4)
        )


In [ ]:
# ============================================================
# MATRIZES DE CONFUSÃO — BASE DE TESTE
# Dynamic K-Means e DenStream
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

for nome in ["Dynamic K-Means", "DenStream (micro)"]:

    # Predições somente da base de teste
    pred_teste = np.asarray(predicoes[nome])[N_CAL:]

    assert len(pred_teste) == len(yteste)

    matriz = pd.crosstab(
        pd.Series(yteste, name="Ocupação real"),
        pd.Series(pred_teste, name="Cluster estimado"),
        margins=True,
        margins_name="Total"
    )

    print(f"\n{'=' * 60}")
    print(f"MATRIZ DE CONFUSÃO — {nome}")
    print(f"{'=' * 60}")
    print(matriz.to_string())

    print(
        "\nARI teste:",
        round(adjusted_rand_score(yteste, pred_teste), 4)
    )
    print("Clusters no teste:", len(np.unique(pred_teste)))


In [ ]:
import matplotlib.pyplot as plt

for nome in ["Dynamic K-Means", "DenStream (micro)"]:

    pred_teste = np.asarray(predicoes[nome])[N_CAL:]

    matriz = pd.crosstab(
        pd.Series(yteste, name="Ocupação real"),
        pd.Series(pred_teste, name="Cluster estimado")
    )

    fig, ax = plt.subplots(
        figsize=(max(6, len(matriz.columns) * 1.1), 3.5)
    )

    im = ax.imshow(
        matriz.to_numpy(),
        aspect="auto",
        cmap="Blues"
    )

    ax.set_xticks(range(len(matriz.columns)))
    ax.set_xticklabels(matriz.columns)

    ax.set_yticks(range(len(matriz.index)))
    ax.set_yticklabels(matriz.index)

    ax.set_xlabel("Cluster estimado")
    ax.set_ylabel("Ocupação real")
    ax.set_title(f"Matriz de confusão — {nome}")

    for i in range(len(matriz.index)):
        for j in range(len(matriz.columns)):
            valor = int(matriz.iloc[i, j])
            cor_texto = "white" if valor > matriz.to_numpy().max() * 0.5 else "black"

            ax.text(
                j, i, str(valor),
                ha="center",
                va="center",
                color=cor_texto
            )

    plt.colorbar(im, ax=ax, label="Frequência")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 11. ABLACÃO DO DECAIMENTO
# ============================================================

print("\n=== Com e sem decaimento ===")

res_sem = avaliar_dynamic_kmeans(
    df_base,
    lotes_flux,
    lambda_decay=0.0,
    fator_limiar=float(esc_dyn["alpha"]),
    n_clusters=K_ESC
)

pred_sem = np.asarray(res_sem["y_pred_stream"])

ablacoes = {
    "Com decaimento": pred_dyn,
    "Sem decaimento": pred_sem
}

linhas_ab = []

for nome, pred in ablacoes.items():

    m = resumir_janelas(
        Xteste,
        yteste,
        pred[N_CAL:]
    )

    linhas_ab.append({
        "Método": nome,
        "Sil_teste": m["silhueta"],
        "Validas_teste": m["frac_sil_valida"],
        "ARI_teste": adjusted_rand_score(
            yteste, pred[N_CAL:]
        ),
        "Clusters_finais": (
            res_dyn["Numero_clusters_final"]
            if nome == "Com decaimento"
            else res_sem["Numero_clusters_final"]
        )
    })

tabela_ablacao = pd.DataFrame(linhas_ab)

print(tabela_ablacao.round(4).to_string(index=False))

# Silhueta pareada na ablação.
ab_com = metricas_janelas(
    Xteste,
    yteste,
    pred_dyn[N_CAL:]
)

ab_sem = metricas_janelas(
    Xteste,
    yteste,
    pred_sem[N_CAL:]
)

comparacao_ab = pd.DataFrame({
    "com": ab_com["silhueta"],
    "sem": ab_sem["silhueta"]
}).dropna()

print("\nJanelas válidas em ambas as versões:",
      len(comparacao_ab))

if not comparacao_ab.empty:
    dif = comparacao_ab["com"] - comparacao_ab["sem"]

    print("Silhueta média com:", comparacao_ab["com"].mean())
    print("Silhueta média sem:", comparacao_ab["sem"].mean())
    print("Diferença pareada:", dif.mean())
    print("Fração de vantagens:", (dif > 0).mean())


In [ ]:
# ============================================================
# 12. TAMANHO E COMPOSIÇÃO DOS CLUSTERS
# ============================================================

print("\n=== Tamanho e composição dos clusters ===")

tabelas_composicao = {}

for nome, pred in predicoes.items():

    dados = pd.DataFrame({
        "Cluster": pred[N_CAL:],
        "Occupancy": yteste
    })

    tab = dados.groupby("Cluster").agg(
        n=("Occupancy", "size"),
        ocupados=("Occupancy", "sum"),
        proporcao_ocupados=("Occupancy", "mean")
    )

    tab["proporcao_total"] = (
        tab["n"] / len(dados)
    )

    tab = tab.sort_values(
        "n",
        ascending=False
    )

    tabelas_composicao[nome] = tab

    print("\n", nome)
    print(tab.round(4).to_string())

    # Cobertura dos clusters quase puros,
    # evitando avaliar somente porcentagens.
    for classe in [0, 1]:
        if classe == 1:
            selecionados = tab[
                tab["proporcao_ocupados"] >= 0.90
            ]
        else:
            selecionados = tab[
                tab["proporcao_ocupados"] <= 0.10
            ]

        print(
            f"Observações em clusters >=90% "
            f"da classe {classe}:",
            int(selecionados["n"].sum())
        )


In [ ]:
# ============================================================
# 13. GRÁFICOS TEMPORAIS
# ============================================================

fig, axes = plt.subplots(
    2, 1, figsize=(13, 8), sharex=True
)

for nome, d in curvas.items():
    for trecho in ["cal", "teste"]:
        z = d[trecho]

        axes[0].plot(
            z["fim"],
            z["silhueta"],
            marker="o",
            markersize=3,
            label=nome if trecho == "cal" else None
        )

        axes[1].plot(
            z["fim"],
            z["ARI_informativo"],
            marker="o",
            markersize=3,
            label=nome if trecho == "cal" else None
        )

for ax in axes:
    ax.axvline(
        N_CAL,
        linestyle="--",
        color="black",
        label="Início do teste"
    )
    ax.grid(alpha=0.25)
    ax.legend()

axes[0].set_ylabel("Silhueta")
axes[1].set_ylabel("ARI informativo")
axes[1].set_xlabel("Observações do fluxo")

axes[0].set_title(
    "Desempenho por janela — Occupancy"
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 14. EVOLUÇÃO DO NÚMERO DE CLUSTERS
# ============================================================

fig, ax = plt.subplots(figsize=(12, 4.5))

for nome, r, cor, legenda in [
    ("Dynamic K-Means", res_dyn, "red", "Método proposto"),
    ("DenStream (p-micro)", res_den, "blue", "DenStream (p-micro)")
]:
    hist = pd.DataFrame(r["ARI_lotes"])

    # Usa a posição real do final de cada lote.
    fins = np.cumsum([
        len(lote) for lote in lotes_flux
    ])

    ax.step(
        fins,
        hist["Numero_clusters"],
        where="post",
        label=legenda,
        color=cor
    )

ax.axhline(
    K_ESC,
    linestyle=":",
    color="green",
    label="K-Means estático"
)

ax.axvline(
    N_CAL,
    color="black",
    linestyle="--",
    label="Início do teste"
)

ax.set_xlabel("Observações do fluxo")
ax.set_ylabel("Número de estruturas")
ax.set_title("Evolução dos agrupamentos")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 15. DISTRIBUIÇÕES POR CLASSE E PERÍODO
# ============================================================

print("\n=== Comparação dos atributos por classe ===")

base_cal = pd.DataFrame(
    Xcal,
    columns=COLUNAS
)
base_cal["Occupancy"] = ycal
base_cal["Trecho"] = "Calibração"

base_teste = pd.DataFrame(
    Xteste,
    columns=COLUNAS
)
base_teste["Occupancy"] = yteste
base_teste["Trecho"] = "Teste"

dados_atributos = pd.concat(
    [base_cal, base_teste],
    ignore_index=True
)

perfil_classe = dados_atributos.groupby(
    ["Trecho", "Occupancy"]
)[COLUNAS].agg(["mean", "std", "median", "count"])

print(perfil_classe.round(3).to_string())

# Gráficos de distribuição condicional:
# calibração versus teste, dentro de cada classe.
fig, axes = plt.subplots(
    len(COLUNAS), 2,
    figsize=(12, 3.1 * len(COLUNAS))
)

for i, coluna in enumerate(COLUNAS):
    for classe in [0, 1]:

        ax = axes[i, classe]

        for trecho in ["Calibração", "Teste"]:

            z = dados_atributos.loc[
                (dados_atributos["Trecho"] == trecho)
                &
                (dados_atributos["Occupancy"] == classe),
                coluna
            ].dropna()

            # Histograma de densidade para permitir
            # comparação com amostras de tamanhos distintos.
            ax.hist(
                z,
                bins=35,
                density=True,
                histtype="step",
                linewidth=1.6,
                label=trecho
            )

        ax.set_title(
            f"{coluna} — Occupancy={classe}"
        )
        ax.set_xlabel("Valor padronizado")
        ax.set_ylabel("Densidade")
        ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 16. COMPOSIÇÃO TEMPORAL DA OCUPAÇÃO
# ============================================================

fig, ax = plt.subplots(figsize=(12, 3.5))

for trecho, z in diag_ocupacao.groupby("trecho"):

    deslocamento = (
        0 if trecho == "Calibração" else N_CAL
    )

    ax.plot(
        z["fim"] + deslocamento,
        z["frac_ocupada"],
        marker=".",
        label=trecho
    )

ax.axvline(
    N_CAL,
    color="black",
    linestyle="--"
)

ax.set_xlabel("Observações do fluxo")
ax.set_ylabel("Proporção ocupada")
ax.set_title("Ocupação ao longo do tempo")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 17. EXPORTAR RESULTADOS
# ============================================================

arquivos = {
    "occupancy_k.csv": tabela_k,
    "occupancy_grade_dynamic.csv": grade_dyn,
    "occupancy_grade_denstream.csv": grade_den,
    "occupancy_resultado_final.csv": tabela_final,
    "occupancy_ablacao.csv": tabela_ablacao,
    "occupancy_janelas_comuns_teste.csv": sil_teste_comuns,
    "occupancy_perfil_por_classe.csv": perfil_classe,
    "occupancy_janelas_ocupacao.csv": diag_ocupacao
}

for nome, tabela in arquivos.items():
    tabela.to_csv(nome)

for nome, tab in tabelas_composicao.items():
    arquivo = (
        "occupancy_clusters_"
        + nome.lower()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("(", "")
            .replace(")", "")
        + ".csv"
    )
    tab.to_csv(arquivo)

print("\nExperimento finalizado.")
print("Resultados salvos em CSV.")


## 8. Diagnósticos exploratórios da Occupancy

Estatísticas descritivas e janelas de avaliação.

In [ ]:
# ============================================================
# OCCUPANCY DETECTION
# ANÁLISE EXPLORATÓRIA + DIAGNÓSTICO FINAL
#
# Executar após o experimento anterior.
# Não modifica os algoritmos nem suas predições.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

COLS_EDA = [
    "Temperature", "Light", "CO2",
    "Humidity", "HumidityRatio"
]


In [ ]:
# ============================================================
# 1. PREPARAÇÃO
# ============================================================

dados_eda = df.copy()

dados_eda["date"] = pd.to_datetime(
    dados_eda["date"]
)

dados_eda = dados_eda.sort_values(
    "date"
).reset_index(drop=True)

dados_eda["hora"] = dados_eda["date"].dt.hour
dados_eda["dia_semana"] = dados_eda["date"].dt.dayofweek

nomes_dias = [
    "Seg", "Ter", "Qua", "Qui",
    "Sex", "Sáb", "Dom"
]

dados_eda["dia_nome"] = (
    dados_eda["dia_semana"].map(
        dict(enumerate(nomes_dias))
    )
)

# Obter o conjunto de teste com datas.
dados_teste_eda = (
    dados_eda[
        dados_eda["origem"] == ARQ_TESTE
    ].copy().reset_index(drop=True)
)

dados_cal_eda = (
    dados_eda[
        dados_eda["origem"] == ARQ_CAL
    ].copy().reset_index(drop=True)
)

# Verificação de alinhamento com as predições.
assert len(dados_cal_eda) == N_CAL
assert len(dados_teste_eda) == len(yteste)

assert np.array_equal(
    dados_teste_eda["Occupancy"].to_numpy(),
    yteste
)

for nome, pred in predicoes.items():
    assert len(pred) == len(fluxo)

print("Observações:", len(dados_eda))
print("Período inicial:", dados_eda["date"].min())
print("Período final:", dados_eda["date"].max())
print("Valores ausentes:", dados_eda.isna().sum().sum())


In [ ]:
# ============================================================
# 2. DESCRIÇÃO GERAL
# ============================================================

print("\n=== Estatísticas descritivas ===")

descricao = dados_eda[COLS_EDA].describe().T

print(
    descricao.round(3).to_string()
)

print("\n=== Ocupação por conjunto ===")

tabela_ocupacao = []

for nome, dados in [
    ("Inicialização", base),
    ("Calibração", cal),
    ("Teste", teste)
]:

    tabela_ocupacao.append({
        "Conjunto": nome,
        "N": len(dados),
        "Desocupada": int(
            (dados["Occupancy"] == 0).sum()
        ),
        "Ocupada": int(
            (dados["Occupancy"] == 1).sum()
        ),
        "Proporção ocupada": (
            dados["Occupancy"].mean()
        )
    })

tabela_ocupacao = pd.DataFrame(
    tabela_ocupacao
)

print(
    tabela_ocupacao.round(4).to_string(
        index=False
    )
)


In [ ]:
# ============================================================
# 3. OCUPAÇÃO POR HORA
# ============================================================

ocupacao_hora = (
    dados_eda.groupby("hora")["Occupancy"]
    .agg(["mean", "size"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(
    ocupacao_hora["hora"],
    ocupacao_hora["mean"]
)

ax.set_xticks(range(24))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Proporção de ocupação")
ax.set_title("Ocupação da sala por hora")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4. OCUPAÇÃO POR DIA DA SEMANA
# ============================================================

ocupacao_dia = (
    dados_eda.groupby("dia_semana")["Occupancy"]
    .agg(["mean", "size"])
    .reindex(range(7))
)

fig, ax = plt.subplots(figsize=(9, 4))

ax.bar(
    nomes_dias,
    ocupacao_dia["mean"]
)

ax.set_xlabel("Dia da semana")
ax.set_ylabel("Proporção de ocupação")
ax.set_title("Ocupação por dia da semana")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 5. MAPA DE CALOR — DIA X HORA
# ============================================================

matriz_ocupacao = dados_eda.pivot_table(
    index="dia_semana",
    columns="hora",
    values="Occupancy",
    aggfunc="mean"
).reindex(
    index=range(7),
    columns=range(24)
)

fig, ax = plt.subplots(figsize=(12, 4.5))

im = ax.imshow(
    matriz_ocupacao.to_numpy(),
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="YlOrRd"
)

ax.set_yticks(range(7))
ax.set_yticklabels(nomes_dias)
ax.set_xticks(range(24))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Dia da semana")
ax.set_title("Proporção de ocupação por dia e hora")

plt.colorbar(
    im,
    ax=ax,
    label="Proporção de ocupação"
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 6. DISTRIBUIÇÃO DAS VARIÁVEIS POR OCUPAÇÃO
# ============================================================

fig, axes = plt.subplots(
    1, len(COLS_EDA),
    figsize=(16, 4)
)

for ax, coluna in zip(axes, COLS_EDA):

    valores = [
        dados_eda.loc[
            dados_eda["Occupancy"] == classe,
            coluna
        ].to_numpy()
        for classe in [0, 1]
    ]

    ax.boxplot(
        valores,
        tick_labels=["Não", "Sim"],
        showfliers=False
    )

    ax.set_title(coluna)
    ax.set_xlabel("Ocupada")
    ax.grid(axis="y", alpha=0.2)

plt.suptitle(
    "Distribuição dos atributos por estado de ocupação"
)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 7. CORRELAÇÃO ENTRE OS ATRIBUTOS
# ============================================================

corr = dados_eda[COLS_EDA].corr()

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    corr,
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(range(len(COLS_EDA)))
ax.set_yticks(range(len(COLS_EDA)))

ax.set_xticklabels(
    COLS_EDA,
    rotation=45,
    ha="right"
)
ax.set_yticklabels(COLS_EDA)

for i in range(len(COLS_EDA)):
    for j in range(len(COLS_EDA)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

plt.colorbar(im, ax=ax)
ax.set_title("Correlação entre os atributos")

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# 7. CORRELAÇÃO ENTRE OS ATRIBUTOS — APENAS INICIALIZAÇÃO
# ============================================================

COLS_EDA = [
    "Temperature", "Light", "CO2",
    "Humidity", "HumidityRatio"
]

# Utilizar somente as observações iniciais (X0)
dados_corr = base[COLS_EDA].copy()

# Matriz de correlação de Pearson
corr = dados_corr.corr(method="pearson")

print(f"Observações utilizadas: {len(dados_corr)}")
print("\nMatriz de correlação:")
print(corr.round(4))

print(
    "\nCorrelação entre Humidity e HumidityRatio:",
    round(corr.loc["Humidity", "HumidityRatio"], 4)
)

# Gráfico
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    corr,
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(range(len(COLS_EDA)))
ax.set_yticks(range(len(COLS_EDA)))

ax.set_xticklabels(
    COLS_EDA,
    rotation=45,
    ha="right"
)
ax.set_yticklabels(COLS_EDA)

for i in range(len(COLS_EDA)):
    for j in range(len(COLS_EDA)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=10
        )

plt.colorbar(im, ax=ax, label="Correlação de Pearson")

ax.set_title(
    "Correlação entre os atributos — Inicialização"
)

plt.tight_layout()

plt.savefig(
    "correlacao_inicializacao.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# 8. DEFINIR JANELAS TEMPORAIS DO TESTE
# ============================================================

# Usar as mesmas janelas do experimento anterior.
# Nenhuma janela atravessa a divisão cal/teste.

janelas_teste = [
    (i, i + JANELA)
    for i in range(
        0,
        len(yteste) - JANELA + 1,
        PASSO
    )
]

# Aqui queremos identificar:
# - se o período contém ocupação
# - quantas classes reais aparecem
# - quantos clusters cada algoritmo identifica
# - se a silhueta é definida

diagnosticos = []

for numero, (ini, fim) in enumerate(
    janelas_teste, start=1
):

    yj = yteste[ini:fim]
    Xj = Xteste[ini:fim]

    proporcao_ocupada = float(
        np.mean(yj == 1)
    )

    n_classes = len(np.unique(yj))

    inicio_data = (
        dados_teste_eda["date"].iloc[ini]
    )
    fim_data = (
        dados_teste_eda["date"].iloc[fim - 1]
    )

    for nome, pred in predicoes.items():

        pj = np.asarray(
            pred[N_CAL + ini:N_CAL + fim]
        )

        n_clusters = len(np.unique(pj))

        sil = np.nan
        if 2 <= n_clusters < len(pj):
            sil = silhouette_score(Xj, pj)

        ari = np.nan
        if n_classes >= 2 and n_clusters >= 2:
            ari = adjusted_rand_score(yj, pj)

        diagnosticos.append({
            "Janela": numero,
            "Método": nome,
            "Início": inicio_data,
            "Fim": fim_data,
            "N": len(yj),
            "Prop_ocupada": proporcao_ocupada,
            "Classes_reais": n_classes,
            "Clusters": n_clusters,
            "Silhueta": sil,
            "ARI_informativo": ari,
            "Um_cluster": n_clusters == 1,
            "Silhueta_valida": np.isfinite(sil),
            "Somente_desocupada": (
                np.all(yj == 0)
            ),
            "Somente_ocupada": (
                np.all(yj == 1)
            ),
            "Duas_classes": n_classes == 2
        })

diagnostico = pd.DataFrame(
    diagnosticos
)


In [ ]:
# ============================================================
# 9. TABELA PRINCIPAL: UM CLUSTER X OCUPAÇÃO
# ============================================================

print(
    "\n=== Relação entre um cluster e ocupação ==="
)

for nome, d in diagnostico.groupby("Método"):

    print(f"\n{nome}")

    tabela = pd.crosstab(
        d["Um_cluster"],
        d["Somente_desocupada"],
        rownames=["Um único cluster"],
        colnames=["Somente desocupada"]
    )

    print(tabela)

    apenas_um = d[d["Um_cluster"]]

    if len(apenas_um) > 0:

        frac = (
            apenas_um["Somente_desocupada"].mean()
        )

        print(
            "Entre janelas com um único cluster, "
            "fração totalmente desocupada:",
            round(frac, 4)
        )

        print(
            "Ocupação média nessas janelas:",
            round(
                apenas_um["Prop_ocupada"].mean(),
                4
            )
        )

    else:
        print(
            "O método não produziu janelas "
            "com um único cluster."
        )


In [ ]:
# ============================================================
# 10. RESUMO DOS DIAGNÓSTICOS
# ============================================================

resumo_diagnostico = (
    diagnostico.groupby("Método")
    .agg(
        Janelas=("Janela", "size"),
        Janelas_um_cluster=("Um_cluster", "sum"),
        Frac_um_cluster=("Um_cluster", "mean"),
        Frac_sil_valida=("Silhueta_valida", "mean"),
        Ocupacao_media=("Prop_ocupada", "mean"),
        Frac_ARI_informativo=(
            "ARI_informativo",
            lambda x: x.notna().mean()
        )
    )
)

print("\n=== Resumo dos diagnósticos ===")

print(
    resumo_diagnostico.round(4).to_string()
)


In [ ]:
# ============================================================
# 11. TABELA DETALHADA POR JANELA
# ============================================================

# Uma linha por período, com o estado real
# e a quantidade de clusters de cada método.

tabela_janelas = diagnostico.pivot(
    index="Janela",
    columns="Método",
    values="Clusters"
)

informacoes = (
    diagnostico
    .drop_duplicates("Janela")
    .set_index("Janela")[
        [
            "Início",
            "Fim",
            "Prop_ocupada",
            "Classes_reais"
        ]
    ]
)

tabela_janelas = informacoes.join(
    tabela_janelas
)

print("\n=== Diagnóstico por janela ===")

print(
    tabela_janelas.round(3).to_string()
)


In [ ]:
# ============================================================
# 12. GRÁFICO: OCUPAÇÃO X NÚMERO DE CLUSTERS
# ============================================================

fig, axes = plt.subplots(
    2, 1,
    figsize=(13, 7),
    sharex=True
)

real = (
    diagnostico
    .drop_duplicates("Janela")
    .sort_values("Janela")
)

axes[0].bar(
    real["Janela"],
    real["Prop_ocupada"],
    width=0.8
)

axes[0].set_ylabel("Proporção ocupada")
axes[0].set_ylim(0, 1)
axes[0].set_title(
    "Ocupação e diversidade dos agrupamentos por janela"
)

for nome, d in diagnostico.groupby("Método"):

    d = d.sort_values("Janela")

    axes[1].plot(
        d["Janela"],
        d["Clusters"],
        marker="o",
        label=nome
    )

axes[1].set_ylabel(
    "Clusters distintos na janela"
)
axes[1].set_xlabel("Janela temporal")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 13. RESUMO CRUZADO:
# TIPO DE JANELA X QUANTIDADE DE CLUSTERS
# ============================================================

def classificar_janela(linha):

    if linha["Somente_desocupada"]:
        return "Somente desocupada"

    if linha["Somente_ocupada"]:
        return "Somente ocupada"

    return "Mista"


diagnostico["Tipo_janela"] = diagnostico.apply(
    classificar_janela,
    axis=1
)

print("\n=== Tipo de janela x clusters ===")

for nome, d in diagnostico.groupby("Método"):

    print(f"\n{nome}")

    tab = pd.crosstab(
        d["Tipo_janela"],
        d["Clusters"]
    )

    print(tab.to_string())


In [ ]:
# ============================================================
# 14. EXPORTAÇÃO
# ============================================================

descricao.to_csv(
    "occupancy_descritivas.csv"
)

tabela_ocupacao.to_csv(
    "occupancy_proporcao_classes.csv",
    index=False
)

ocupacao_hora.to_csv(
    "occupancy_por_hora.csv",
    index=False
)

ocupacao_dia.to_csv(
    "occupancy_por_dia.csv"
)

corr.to_csv(
    "occupancy_correlacoes.csv"
)

diagnostico.to_csv(
    "occupancy_diagnostico_janelas.csv",
    index=False
)

resumo_diagnostico.to_csv(
    "occupancy_resumo_diagnostico.csv"
)

tabela_janelas.to_csv(
    "occupancy_clusters_por_janela.csv"
)

print("\nAnálise exploratória e diagnóstico concluídos.")


In [ ]:
# Verificação final — Dynamic K-Means

historico = pd.DataFrame(res_dyn["ARI_lotes"])

contagens = historico["Numero_clusters"].to_numpy()

print("Clusters iniciais:", K_ESC)
print("Clusters finais:", res_dyn["Numero_clusters_final"])

print(
    "Histórico não decrescente:",
    bool(np.all(np.diff(contagens) >= 0))
)

print(
    "Clusters adicionais:",
    res_dyn["Numero_clusters_final"] - K_ESC
)

print(
    "IDs encontrados no teste:",
    np.unique(pred_dyn[N_CAL:])
)

print(
    "IDs que não receberam observações no teste:",
    sorted(
        set(range(res_dyn["Numero_clusters_final"]))
        - set(np.unique(pred_dyn[N_CAL:]))
    )
)

assert np.all(np.diff(contagens) >= 0)
assert contagens[-1] == res_dyn["Numero_clusters_final"]
assert len(pred_dyn) == len(fluxo)
assert len(np.unique(pred_dyn[N_CAL:])) == 6


## 9. Verificação offline na Occupancy

Rotina complementar usando o modelo final selecionado na calibração.

Esta etapa não substitui os resultados com p-micro-clusters. Usa a implementação nativa predict_one do River, que agrupa os p-micro-clusters por conectividade de densidade antes de atribuir cada ponto ao macro-centro mais próximo.

Protocolo: parâmetros congelados após a calibração; inicialização em X0; avaliação do lote com o modelo anterior à atualização; atualização com os pontos do lote; alinhamento de IDs de macro-clusters entre lotes. A comparação com o Dynamic K-Means ainda apresenta diferenças no instante da atribuição (pré-lote versus atualizações dentro do lote).


In [ ]:
from river import cluster as river_cluster
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment


def avaliar_denstream_offline(df_base, lotes, config):
    """River offline nativo; pré-lote; IDs persistentes via casamento de centros."""
    X0_df = obter_variaveis(df_base)
    colunas = list(X0_df.columns)
    modelo = river_cluster.DenStream(**config)
    for x in X0_df.to_dict(orient="records"):
        modelo.learn_one(x)

    ids_prev = np.array([], dtype=int)
    centros_prev = np.empty((0, len(colunas)))
    proximo_id = 0
    predicoes = []
    historico = []
    y_true = []

    for indice_lote, lote in enumerate(lotes, 1):
        xl_df = obter_variaveis(lote)
        X = xl_df.to_numpy(float)
        registros = xl_df.to_dict(orient="records")
        y = lote["Grupo"].to_numpy()

        # Chamada nativa dispara a fase offline do River para o estado ATUAL.
        # Os rótulos nativos são reatribuídos a cada novo reclustering.
        modelo.predict_one(registros[0])
        centros_dict = modelo.centers  # macro-centros calculados pelo River
        chaves = list(centros_dict)
        centros = np.array([
            [centros_dict[k].get(c, 0.0) for c in colunas]
            for k in chaves
        ], dtype=float).reshape(-1, len(colunas))
        n_macros_antes = len(centros)

        if n_macros_antes == 0:
            pred = np.full(len(X), -1, dtype=int)
        else:
            # Alinhamento de macro-IDs entre lotes para evitar renumeração arbitrária.
            ids = np.full(n_macros_antes, -1, dtype=int)
            if len(centros_prev):
                dist = cdist(centros, centros_prev)
                rr, cc = linear_sum_assignment(dist)
                for r, c in zip(rr, cc):
                    if dist[r, c] <= 2.0 * config["epsilon"]:
                        ids[r] = ids_prev[c]
            for i in range(len(ids)):
                if ids[i] < 0:
                    ids[i] = proximo_id
                    proximo_id += 1
            pred = ids[cdist(X, centros).argmin(axis=1)]
            centros_prev, ids_prev = centros.copy(), ids.copy()

        predicoes.extend(pred)
        y_true.extend(y)
        for registro in registros:
            modelo.learn_one(registro)
        historico.append({
            "Lote": indice_lote,
            "Numero_macro_antes": n_macros_antes,
            "Numero_micro_depois": len(modelo.p_micro_clusters),
        })

    # Macro-clusters após TODO o fluxo, sem contaminar predições passadas.
    if len(predicoes):
        # Usar um registro conhecido somente para solicitar reclustering, sem aprendizado.
        modelo.predict_one(registros[-1])
        n_macro_final = modelo.n_clusters
    else:
        n_macro_final = 0
    return {
        "y_true_stream": np.array(y_true),
        "y_pred_stream": np.array(predicoes),
        "Numero_macro_final": n_macro_final,
        "historico": pd.DataFrame(historico),
        "observacao": "ARI global condicionado ao alinhamento de IDs por centros"
    }

res_offline = avaliar_denstream_offline(df_base, lotes_flux, cfg_den)
pred_offline = res_offline["y_pred_stream"]
assert len(pred_offline) == len(Xflux)

# Reaproveitar exatamente as funções de avaliação já definidas.
met = resumir_janelas(Xteste, yteste, pred_offline[N_CAL:])
resumo_offline = pd.DataFrame([
    {
        "Método": "DenStream (micro)",
        "Silhueta teste": tabela_final.set_index("Método").loc["DenStream (micro)", "Sil_teste"],
        "Janelas válidas": tabela_final.set_index("Método").loc["DenStream (micro)", "Validas_teste"],
        "ARI teste": tabela_final.set_index("Método").loc["DenStream (micro)", "ARI_teste"],
        "Estruturas finais": res_den["Numero_clusters_final"]
    },
    {
        "Método": "DenStream (macro/offline)",
        "Silhueta teste": met["silhueta"],
        "Janelas válidas": met["frac_sil_valida"],
        "ARI teste": adjusted_rand_score(yteste, pred_offline[N_CAL:]),
        "Estruturas finais": res_offline["Numero_macro_final"]
    }
])
print(resumo_offline.round(4).to_string(index=False))
print("IDs macro distintos no teste:", np.unique(pred_offline[N_CAL:]))
print("Macros por lote (resumo):")
print(res_offline["historico"].describe().round(2))
resumo_offline.to_csv("occupancy_denstream_offline.csv", index=False)

# Silhueta pareada (somente janelas válidas em ambas as variantes).
micro_w = metricas_janelas(Xteste, yteste, pred_den[N_CAL:])
macro_w = metricas_janelas(Xteste, yteste, pred_offline[N_CAL:])
w = pd.DataFrame({"micro": micro_w["silhueta"], "macro": macro_w["silhueta"]}).dropna()
print("Janelas comuns micro/macro:", len(w), "de", len(micro_w))
if len(w):
    print("Silhueta média pareada:", w.mean().round(4).to_dict())
    print("Diferença macro - micro:", round(float((w["macro"] - w["micro"]).mean()), 4))


In [ ]:
# ============================================================
# DENSTREAM — COMPARAÇÃO FINAL ONLINE VS OFFLINE
# Base completa: inicialização + calibração + teste
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# 1. Mesmos hiperparâmetros da configuração final
cfg_offline_final = config_den(
    esc_den["epsilon"],
    esc_den["mu"],
    esc_den["meia_vida"]
)
cfg_offline_final["beta"] = float(
    esc_den["beta"] if "beta" in esc_den.index else 0.75
)

print("=== CONFIGURAÇÃO UTILIZADA ===")
print(cfg_offline_final)

# 2. Executar o DenStream offline
# A função reinicializa o modelo em df_base
# e processa todos os lotes de lotes_flux.
res_offline_final = avaliar_denstream_offline(
    df_base,
    lotes_flux,
    cfg_offline_final
)

# 3. Recuperar as predições online e offline
pred_online = np.asarray(
    predicoes["DenStream (micro)"]
)

pred_offline = np.asarray(
    res_offline_final["y_pred_stream"]
)

assert len(pred_online) == len(pred_offline)
assert len(pred_online) == N_CAL + len(yteste)

# 4. Avaliar separadamente calibração e teste
comparacao = []

for etapa, inicio, fim, X, y in [
    ("Calibração", 0, N_CAL, Xcal, ycal),
    ("Teste", N_CAL, len(pred_online), Xteste, yteste)
]:
    micro = pred_online[inicio:fim]
    macro = pred_offline[inicio:fim]

    for nome, pred in [
        ("Online (micro)", micro),
        ("Offline (macro)", macro)
    ]:
        m = resumir_janelas(X, y, pred)

        comparacao.append({
            "Conjunto": etapa,
            "Fase": nome,
            "ARI": adjusted_rand_score(y, pred),
            "Silhueta": m["silhueta"],
            "Janelas_validas": m["frac_sil_valida"],
            "Clusters_distintos": len(np.unique(pred))
        })

tabela_online_offline = pd.DataFrame(comparacao)

print("\n=== COMPARAÇÃO ONLINE VS OFFLINE ===")
print(
    tabela_online_offline.round(4).to_string(index=False)
)

# 5. Comparar as partições diretamente,
# independentemente dos números dos clusters.
print("\n=== CONCORDÂNCIA ENTRE AS DUAS FASES ===")

for etapa, inicio, fim in [
    ("Calibração", 0, N_CAL),
    ("Teste", N_CAL, len(pred_online))
]:
    ari_entre_fases = adjusted_rand_score(
        pred_online[inicio:fim],
        pred_offline[inicio:fim]
    )

    print(
        f"{etapa}: ARI online vs offline = "
        f"{ari_entre_fases:.4f}"
    )

# 6. Quantidade de estruturas ao fim do fluxo
print("\n=== AGRUPAMENTOS FINAIS ===")
print(
    "Micro-clusters potenciais:",
    res_den["Numero_clusters_final"]
)
print(
    "Macroclusters offline:",
    res_offline_final["Numero_macro_final"]
)

# 7. Matrizes de confusão no teste
for nome, pred in [
    ("Online", pred_online[N_CAL:]),
    ("Offline", pred_offline[N_CAL:])
]:
    print(f"\n=== MATRIZ DE CONFUSÃO — {nome} ===")

    matriz = pd.crosstab(
        pd.Series(np.asarray(yteste), name="Ocupação real"),
        pd.Series(pred, name="Cluster estimado")
    )

    print(matriz.to_string())

# 8. Salvar os resultados
tabela_online_offline.to_csv(
    "denstream_online_offline_final.csv",
    index=False
)
